# Silver Layer — Supply Chain Data Platform

Transforms Bronze raw tables into trusted, validated, and conformed Silver tables.

| Bronze Source | Silver Target | Key Transformations |
|---|---|---|
| `bronze_dim_products` | `silver_dim_products` | Dedup, null filter, trim, INITCAP category, price_tier |
| `bronze_dim_warehouses` | `silver_dim_warehouses` | Dedup, null filter, trim, INITCAP location |
| `bronze_fact_orders` | `silver_fact_orders` | Dedup, null filter, UPPER status, derive order_date |
| `bronze_fact_shipments` | `silver_fact_shipments` | Dedup, null filter, INITCAP carrier, derive ship_date |

All Silver tables land in `abd_supplychain_dev.silver`.

In [0]:
-- Create the Silver schema if it doesn't already exist
CREATE SCHEMA IF NOT EXISTS abd_supplychain_dev.silver;

In [0]:
-- Silver: dim_products
-- Deduplication on product_id (latest _ingested_at wins)
-- Nulls filtered on primary key & unit_price
-- Strings trimmed, category INITCAP-normalized
-- Derived: price_tier (LOW / MEDIUM / HIGH)
CREATE OR REPLACE TABLE abd_supplychain_dev.silver.silver_dim_products AS
SELECT
  product_id,
  TRIM(sku)                          AS sku,
  INITCAP(TRIM(category))            AS category,
  unit_price_usd,
  TRIM(product_description)          AS product_description,
  CASE
    WHEN unit_price_usd < 50   THEN 'LOW'
    WHEN unit_price_usd < 200  THEN 'MEDIUM'
    ELSE                            'HIGH'
  END                                AS price_tier,
  current_timestamp()                AS _silver_processed_at
FROM (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY _ingested_at DESC) AS rn
  FROM abd_supplychain_dev.default.bronze_dim_products
  WHERE product_id      IS NOT NULL
    AND sku             IS NOT NULL
    AND unit_price_usd  >  0
) deduped
WHERE rn = 1;

In [0]:
-- Silver: dim_warehouses
-- Deduplication on warehouse_id (latest _ingested_at wins)
-- Nulls filtered on primary key & capacity
-- Strings trimmed, location INITCAP-normalized
CREATE OR REPLACE TABLE abd_supplychain_dev.silver.silver_dim_warehouses AS
SELECT
  warehouse_id,
  TRIM(warehouse_name)               AS warehouse_name,
  INITCAP(TRIM(location))            AS location,
  max_capacity_units,
  current_timestamp()                AS _silver_processed_at
FROM (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY warehouse_id ORDER BY _ingested_at DESC) AS rn
  FROM abd_supplychain_dev.default.bronze_dim_warehouses
  WHERE warehouse_id        IS NOT NULL
    AND max_capacity_units  >  0
) deduped
WHERE rn = 1;

In [0]:
-- Silver: fact_orders
-- Deduplication on order_id (latest _ingested_at wins)
-- Nulls filtered on all FK columns and timestamp
-- quantity must be positive
-- order_status UPPER-normalized
-- Derived: order_date (DATE extracted from order_timestamp)
CREATE OR REPLACE TABLE abd_supplychain_dev.silver.silver_fact_orders AS
SELECT
  order_id,
  customer_id,
  warehouse_id,
  product_id,
  quantity,
  UPPER(TRIM(order_status))          AS order_status,
  order_timestamp,
  CAST(order_timestamp AS DATE)      AS order_date,
  current_timestamp()                AS _silver_processed_at
FROM (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY _ingested_at DESC) AS rn
  FROM abd_supplychain_dev.default.bronze_fact_orders
  WHERE order_id         IS NOT NULL
    AND customer_id      IS NOT NULL
    AND product_id       IS NOT NULL
    AND warehouse_id     IS NOT NULL
    AND quantity         >  0
    AND order_timestamp  IS NOT NULL
) deduped
WHERE rn = 1;

In [0]:
-- Silver: fact_shipments
-- Deduplication on shipment_id (latest _ingested_at wins)
-- Nulls filtered on primary key, order_id, and timestamp
-- shipment_cost_usd must be >= 0 (free shipments allowed)
-- carrier INITCAP-normalized, delivery_notes trimmed
-- Derived: ship_date (DATE extracted from ship_timestamp)
CREATE OR REPLACE TABLE abd_supplychain_dev.silver.silver_fact_shipments AS
SELECT
  shipment_id,
  order_id,
  INITCAP(TRIM(carrier))             AS carrier,
  shipment_cost_usd,
  ship_timestamp,
  CAST(ship_timestamp AS DATE)       AS ship_date,
  TRIM(delivery_notes)               AS delivery_notes,
  current_timestamp()                AS _silver_processed_at
FROM (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY shipment_id ORDER BY _ingested_at DESC) AS rn
  FROM abd_supplychain_dev.default.bronze_fact_shipments
  WHERE shipment_id        IS NOT NULL
    AND order_id           IS NOT NULL
    AND shipment_cost_usd  >= 0
    AND ship_timestamp     IS NOT NULL
) deduped
WHERE rn = 1;

In [0]:
-- Validation: confirm all Silver tables were populated
SELECT 'silver_dim_products'   AS table_name, COUNT(*) AS row_count FROM abd_supplychain_dev.silver.silver_dim_products
UNION ALL
SELECT 'silver_dim_warehouses' AS table_name, COUNT(*) AS row_count FROM abd_supplychain_dev.silver.silver_dim_warehouses
UNION ALL
SELECT 'silver_fact_orders'    AS table_name, COUNT(*) AS row_count FROM abd_supplychain_dev.silver.silver_fact_orders
UNION ALL
SELECT 'silver_fact_shipments' AS table_name, COUNT(*) AS row_count FROM abd_supplychain_dev.silver.silver_fact_shipments
ORDER BY table_name;

In [0]:
SELECT * FROM abd_supplychain_dev.silver.silver_dim_products
ORDER BY product_id
DESC LIMIT 10